In [1]:
!pip install openai
!pip install langchain
!pip install langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from google.colab import userdata

api_key = userdata.get("kosa")

if api_key is None:
    raise ValueError("OpenAI API KEY가 없습니다.")

os.environ["OPENAI_API_KEY"] = api_key

In [3]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 2.5 MB/s eta 0:00:00


In [4]:
# 대화형 프롬프트 생성
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
# 모델의 출력 >> 구문 분석 >> 원하는 형식으로 변환
from langchain_core.output_parsers import BaseOutputParser

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Act as a classifier that accurately categorizes the sentiment of comments. Given a user-input comment, write '1' if the comment is positive, and '0' if the comment is negative. Output the integer '1' or '0' only, without any other text."),
    ("human", "{input_text}"),

 ])

chat_model = ChatOpenAI(temperature=0)

class classificationOutputParser(BaseOutputParser):
    def parse(self, text: str):
        if '1' in text:
          return 1
        return 0

# LangChain Expression Language(LCEL)
# 사용자 입력에 대해 대화 템플릿 사용
# >> 모델 에게 요청(request)
# >> 모델의 출력을 분류하여 반환

chain = chat_prompt_template | chat_model | classificationOutputParser()

def langchain_llm(input_text):
   output = chain.invoke({"input_text": input_text}) # output 1 또는 0
   return output

In [5]:
def classify_text(input_text):
  output = langchain_llm(input_text)
  return output

분석에 사용할 데이터 로드 >> 분석

In [6]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/tykimos/tykimos.github.io/master/warehouse/dataset/tarr_train.txt",
    filename="tarr_train.txt",
)

('tarr_train.txt', <http.client.HTTPMessage at 0x7a2ba4b620c0>)

In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix

df = pd.read_csv('tarr_train.txt', sep='\t')
df.head()

,id,comment,label
0,1,여기 음식은 언제 와도 실망시키지 않아요. 최고!,1
1,2,여기 라멘 진짜 ㄹㅇ 맛있어요. 국물이 진하고 면도 쫄깃해서 너무 좋았습니다.,1
2,3,"진짜 깔끔하고, 맛도 좋았어요. 추천합니다!",1
3,4,왜 이렇게 유명한지 모르겠음ㅋㅋ ㄹㅈㄷ 맛없음,0
4,5,인생 타르트를 여기서 만났어요❤️ 달지 않고 고소해서 정말 추천합니다!,1


In [8]:
actual_labels =[]
predicted_labels =[]

total = len(df)

for index, row in df.iterrows():
  comment = row['comment']
  actual_label = row['label']

  # llm에 질의 >> 1(긍정), 0(부정) 코멘트 분류
  predicted_label  = classify_text(comment)

  # 실제 분류 데이터와 예측 분류 데이터를 추가함
  actual_labels.append(actual_label)
  predicted_labels.append(predicted_label)

  print(f"[{index+1}]/[{total}]")
  print("comment : ", comment)
  print("actual class : ", actual_label)
  print("predicted class : ", predicted_label)
  print("---------------")

  if index > 10 :
    break

# 정확도 계산
accuracy = accuracy_score(actual_labels, predicted_labels)
print(f"Accuracy: {accuracy*100:.2f}%")

# Confusion matrix 계산
cm = confusion_matrix(actual_labels, predicted_labels, labels=[1, 0])
# Confusion matrix 표현
print("\nConfusion Matrix:")
print("         Predicted:")
print("         긍정    부정")
print("Actual")
print("긍정      {:<5}  {:<5}".format(cm[0][0], cm[0][1]))
print("부정      {:<5}  {:<5}".format(cm[1][0], cm[1][1]))

[1]/[300]
comment :  여기 음식은 언제 와도 실망시키지 않아요. 최고!
actual class :  1
predicted class :  1
---------------
[2]/[300]
comment :  여기 라멘 진짜 ㄹㅇ 맛있어요. 국물이 진하고 면도 쫄깃해서 너무 좋았습니다.
actual class :  1
predicted class :  1
---------------
[3]/[300]
comment :  진짜 깔끔하고, 맛도 좋았어요. 추천합니다!
actual class :  1
predicted class :  1
---------------
[4]/[300]
comment :  왜 이렇게 유명한지 모르겠음ㅋㅋ ㄹㅈㄷ 맛없음
actual class :  0
predicted class :  0
---------------
[5]/[300]
comment :  인생 타르트를 여기서 만났어요❤️ 달지 않고 고소해서 정말 추천합니다!
actual class :  1
predicted class :  1
---------------
[6]/[300]
comment :  메뉴 설명을 너무 친절하게 해주셔서 고르기 수월했어요.
actual class :  1
predicted class :  1
---------------
[7]/[300]
comment :  사진과 음식이 너무 달라서 실망했습니다.
actual class :  0
predicted class :  0
---------------
[8]/[300]
comment :  주변에 추천하려고 사진도 많이 찍었어요. 좋아요!
actual class :  1
predicted class :  1
---------------
[9]/[300]
comment :  솔직히...? 맛이 그닥이에요. 리뷰랑 너무 다르네.
actual class :  0
predicted class :  0
---------------
[10]/[300]
comment :  진짜 개꿀맛..ㅠ 다른곳 안가.
a

In [9]:
# eos